In [1]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Read Data
X_train = pd.read_csv('X_train.csv')
X_test = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
y_test = pd.read_csv('y_test.csv').squeeze()

# Print original shape
print("Original:", X_train.shape, X_test.shape)

# Take a small sample
X_train_small = X_train.iloc[:8000]
y_train_small = y_train.iloc[:8000]

X_test_small = X_test.iloc[:2000]
y_test_small = y_test.iloc[:2000]

print("Small subset:", X_train_small.shape, X_test_small.shape)


Original: (750000, 47) (250000, 47)
Small subset: (8000, 47) (2000, 47)


Due to computational constraints of non-linear SVMs, we trained on a smaller subset of data.

In [3]:
# Create Poly Kernel SVM 
svm_poly_balanced = SVC(
    kernel='poly',
    degree=3,
    C=1.0,
    gamma='scale',
    class_weight='balanced',
    random_state=42
)

# Train
svm_poly_balanced.fit(X_train_small, y_train_small)

SVC(class_weight='balanced', kernel='poly', random_state=42)

In [4]:
# Prediction
y_pred_balanced = svm_poly_balanced.predict(X_test_small)

In [5]:
print("=== Classification Report (Poly Kernel SVM + Balanced Class Weight, Small Data) ===")
print(classification_report(y_test_small, y_pred_balanced))

=== Classification Report (Poly Kernel SVM + Balanced Class Weight, Small Data) ===
              precision    recall  f1-score   support

           0       0.99      0.97      0.98      1978
           1       0.04      0.09      0.05        22

    accuracy                           0.96      2000
   macro avg       0.51      0.53      0.52      2000
weighted avg       0.98      0.96      0.97      2000



Using polynomial kernel SVM with class_weight='balanced', we achieved a high overall accuracy of 96%. However, due to severe class imbalance, the model struggled to detect fraud cases, achieving only 9% recall for the fraud class.Using class_weight alone can not be enough to deal with extreme imbalance,we add SMOTE.

In [6]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

In [7]:
from sklearn.utils import resample

# Stratified sampling (maintaining class proportions)
X_train_small, y_train_small = resample(
    X_train, y_train, 
    n_samples=8000, 
    stratify=y_train, 
    random_state=42
)

In [8]:
# Integrating SMOTE in Pipeline
pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),  
    ('svm', SVC(
        kernel='poly',
        degree=3,
        class_weight='balanced',  
        random_state=42
    ))
])
pipeline.fit(X_train_small, y_train_small)

Pipeline(steps=[('smote', SMOTE(random_state=42)),
                ('svm',
                 SVC(class_weight='balanced', kernel='poly', random_state=42))])

In [14]:
from sklearn.metrics import ConfusionMatrixDisplay

# Prediction
y_pred = pipeline.predict(X_test_small)

# Print classfication report
print("=== Classification Report ===")
print(classification_report(y_test_small, y_pred))



=== Classification Report ===
              precision    recall  f1-score   support

           0       0.99      0.98      0.98      1978
           1       0.02      0.05      0.03        22

    accuracy                           0.97      2000
   macro avg       0.51      0.51      0.51      2000
weighted avg       0.98      0.97      0.97      2000



In [13]:
from sklearn.svm import SVC
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import ADASYN  # Use more advanced oversampling method

# Final Pipeline
final_pipeline = ImbPipeline([
    ('adasyn', ADASYN(random_state=42, n_neighbors=3)),  # replace SMOTE
    ('svm', SVC(
        kernel='poly',
        degree=2,                 # Force quadratic polynomial
        C=50,                    # Significantly increase regularization strength
        gamma=0.1,               # Manually lower kernel coefficient
        class_weight={0:1, 1:30}, # Extremely increase the weight for fraud class
        random_state=42,
        probability=True
    ))
])

# Train and Evaluate
final_pipeline.fit(X_train_small, y_train_small)
y_pred_final = final_pipeline.predict(X_test_small)
print(classification_report(y_test_small, y_pred_final))

              precision    recall  f1-score   support

           0       0.99      0.97      0.98      1978
           1       0.02      0.05      0.03        22

    accuracy                           0.96      2000
   macro avg       0.50      0.51      0.50      2000
weighted avg       0.98      0.96      0.97      2000

